In [1]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.base import clone

import joblib
from pathlib import Path

from prepare_data import prepare_data
from preprocessor_function import preprocessor_function
from models_list import models

In [2]:
data = pd.read_csv('data/train.csv')
data

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


In [3]:
y = np.log1p(data['SalePrice'])
X = prepare_data(data.drop('SalePrice',axis=1))

In [4]:
#удаляем выбросы
X = X[X['GrLivArea'] < 4500] 

In [5]:
#синхронизируем данные
y = y[X.index]

In [ ]:
#делим данные не тестовые и тренировочные
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    shuffle=True,
    randeom_state=42,
)

In [7]:
#создаем списки с категориальными признакиами и нумерик
numeric_colums = X_train.select_dtypes(include=['number']).columns
categorical_colums = X_train.select_dtypes(include=['object', 'string']).columns

In [8]:
#создаем препроцессор
preprocessor = preprocessor_function(X_train, categorical_colums, numeric_colums)

In [9]:
#проверка работоспособности препроцессора

preview_preprocessor = clone(preprocessor).set_output(transform='pandas')
X_encoded_preview = preview_preprocessor.fit_transform(X, y)
X_encoded_preview.head()

,Street_Pave,Alley_Grvl,Alley_Pave,LotShape_IR2,LotShape_IR3,LotShape_Reg,LandContour_HLS,LandContour_Low,LandContour_Lvl,Utilities_NoSeWa,...,PoolArea,MiscVal,MoSold,YrSold,HouseAge,RemodAge,TotalBath,HasGarage,HasBasement,TotalSF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,-0.063709,-0.087748,-1.601578,0.138375,-1.045249,-0.871676,1.174852,0.242536,0.161363,0.011436
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,-0.063709,-0.087748,-0.490155,-0.614427,-0.185182,0.388660,0.386571,0.242536,0.161363,-0.042838
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.063709,-0.087748,0.991743,0.138375,-0.979090,-0.823201,1.174852,0.242536,0.161363,0.192351
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.063709,-0.087748,-1.601578,-1.367230,1.799589,0.631032,-1.189990,0.242536,0.161363,-0.108743
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.063709,-0.087748,2.103167,0.138375,-0.946011,-0.726252,1.174852,0.242536,0.161363,1.015514


In [10]:
from sklearn.metrics import root_mean_squared_error, make_scorer

def rmse_original(y_log, pred_log):
    y = np.expm1(y_log)
    pred = np.expm1(pred_log)
    return root_mean_squared_error(y, pred)

rmse_scorer = make_scorer(rmse_original, greater_is_better=False)

In [11]:
# объявляем массив, куда мы будем записывать лучшие параметры моделей
results = []
best_score = None

output_dir = Path('best_ml_model')
output_dir.mkdir(parents=True, exist_ok=True)

# Запуск цикла обучения 
for name, (model, params) in models.items():
    if name == 'CatBoostRegressor':
        pipe = Pipeline([
            ('model', model),
        ])
        grid = GridSearchCV(pipe, params, cv=5, scoring=rmse_scorer, n_jobs=-1)
        grid.fit(X_train, y_train, model__cat_features = categorical_colums.tolist())
    else:
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model),
        ])
        grid = GridSearchCV(pipe, params, cv=5, scoring=rmse_scorer, n_jobs=-1)
        grid.fit(X_train, y_train)

    test_rmse = root_mean_squared_error(
        np.expm1(y_test),
        np.expm1(grid.predict(X_test)),
    )
    results.append({
        'model': name,
        'best_cv_score': -grid.best_score_,
        'test_score': test_rmse,
        'best_params': grid.best_params_,
        'best_estimator': grid.best_estimator_
    })

    if best_score is None or grid.best_score_ > best_score:
        best_score = grid.best_score_
        joblib.dump(grid.best_estimator_, output_dir / 'best_model.pkl')

print("Обучение завершено. Рассчитываем финальные метрики...")

Обучение завершено. Рассчитываем финальные метрики...


In [12]:
results_df = pd.DataFrame(results).sort_values(
    'test_score',
)

print(pd.DataFrame(results_df[['model', 'best_cv_score', 'test_score', 'best_params']]))

               model  best_cv_score    test_score best_params
0  CatBoostRegressor   22486.391439  29438.673495          {}


In [13]:
model = joblib.load(output_dir / 'best_model.pkl')

In [14]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False


In [21]:
from sklearn.metrics import root_mean_squared_error
if(model.named_steps['model'].__class__.__name__ == 'CatBoostRegressor'):
    model.fit(X,np.expm1(y), model__cat_features = categorical_colums.tolist())
    final_pred = model.predict(X)
else:
    model.fit(X,np.expm1(y))
    final_pred = model.predict(X)

print(final_pred)
rmse = root_mean_squared_error(np.expm1(y), final_pred)
print(f"rmse: {rmse}")


[207016.39003289 174875.33498592 213050.69001853 ... 261051.96683814
 137380.49917991 153279.57195141]
rmse: 10363.807319428874


In [22]:
test_data = pd.read_csv('data/test.csv')
y_pred = model.predict(prepare_data(test_data))

In [23]:
submission = pd.DataFrame({
    "Id": test_data["Id"],
    "SalePrice": y_pred
})

submission_path = 'data/ml_submission/submission.csv'
submission.to_csv(submission_path, index=False)



In [24]:
submission

,Id,SalePrice
0,1461,121276.845666
1,1462,167309.904417
2,1463,186838.263859
3,1464,193215.725235
4,1465,184681.542917
...,...,...
1454,2915,86632.956025
1455,2916,89820.161531
1456,2917,162195.304573
1457,2918,121358.260415
